In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 14. Attention機構

> **学習目標**
> - Q, K, V (database lookup)
> - Scaled Dot-Product Attention $\sqrt{d_k}$ 複雑度
> - Attention 時間/空間 複雑度 $O(n^2 d)$
> - Attention CPU vs GPU 比較

## 14.1 Attention

RNN : . .

Bahdanau Attention(2014): **** .

**Self-Attention** : . RNN 複雑度 学習.

## 14.2 ··値 (Q, K, V) —

データ :
- **Query (Q)**:
- **Key (K)**: ( )
- **Value (V)**:

Attention: Q K 複雑度 計算 → 複雑度 V .

:
$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

- $Q \in \mathbb{R}^{n \times d_k}$: $n$
- $K \in \mathbb{R}^{n \times d_k}$: $n$
- $V \in \mathbb{R}^{n \times d_v}$: $n$ 値
- $QK^\top \in \mathbb{R}^{n \times n}$: -


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Scaled Dot-Product Attention  (NumPy)
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    """Scaled Dot-Product Attention.
    Q, K, V: (n, d) or (batch, n, d)
    """
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)  # (..., n, n)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

#  
np.random.seed(0)
n, d = 4, 8
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

output, weights = attention(Q, K, V)
print(f"Q shape: {Q.shape}")
print(f"Attention output: {output.shape}")
print(f"Attention weights: {weights.shape}")
print(f"\n  Sum (softmax ): {weights.sum(axis=-1).round(4)}")


## 14.3 $\sqrt{d_k}$ ?

$QK^\top$ $\sum_{i=1}^{d_k} Q_i K_i$. Q, K $\mathcal{N}(0, 1)$:

$$\mathrm{Var}(QK^\top_{ij}) = \sum_{i=1}^{d_k} \mathrm{Var}(Q_i K_i) = d_k$$

 = $\sqrt{d_k}$. $d_k$ softmax **** ( 値 1, 0).

$\sqrt{d_k}$ 1 → softmax 分布 . 複雑度 .


In [ ]:
# sqrt(d_k)   
np.random.seed(0)
dks = [8, 64, 512, 2048]
fig, axes = plt.subplots(1, len(dks), figsize=(16, 4))

for ax, d_k in zip(axes, dks):
    n = 4
    Q = np.random.randn(n, d_k)
    K = np.random.randn(n, d_k)
    #  
    scores_no = Q @ K.T
    #  
    scores_scaled = scores_no / np.sqrt(d_k)

    # softmax
    p_no = softmax(scores_no)
    p_scaled = softmax(scores_scaled)

    ax.bar(['q0', 'q1', 'q2', 'q3'], p_no[0], alpha=0.5, label='Scaling X', color='red')
    ax.bar(['q0', 'q1', 'q2', 'q3'], p_scaled[0], alpha=0.5, label='Scaling O', color='blue')
    ax.set_title(f'd_k={d_k}\nVariance: {scores_no.var():.1f} → {scores_scaled.var():.1f}')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/ch14_scaling_effect.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> d_k  Scaling の場合 softmax  Value  ().")


## 14.4 Attention 時間·空間 複雑度

$$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}(QK^\top / \sqrt{d_k}) V$$

- $QK^\top$: $n \times d \times n = O(n^2 d)$
- $\mathrm{softmax}(A) V$: $n \times n \times d = O(n^2 d)$
- ** 時間**: $O(n^2 d)$
- **空間**: $A \in \mathbb{R}^{n \times n}$ → $O(n^2)$

問題: シーケンス長 $n$ $n^2$ . 8K $n^2 = 6.4 \times 10^7$.
→ Flash Attention, Attention (Ch 27).


In [ ]:
# シーケンス長  可視化
seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192]
n2 = [n**2 for n in seq_lens]
n2d = [n**2 * 64 for n in seq_lens]  # d=64

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(seq_lens)), n2, alpha=0.7, label='$n^2$ (Space)')
ax.bar(range(len(seq_lens)), [n/1000 for n in n2d], alpha=0.7, label='$n^2 \\cdot d$ / 1000 (Operation)')
ax.set_xticks(range(len(seq_lens)))
ax.set_xticklabels([str(n) for n in seq_lens])
ax.set_xlabel('Sequence Length n')
ax.set_ylabel('Complexity')
ax.set_title('Attention Complexity: $O(n^2 d)$ — Sequence Length  ')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/ch14_complexity.png', dpi=100, bbox_inches='tight')
plt.show()
print("8K Context 64M  Attention Scores. Flash Attention  .")


## 14.5 (Causal Masking)

GPT autoregressive モデル .

 行列 $M$:
$$M_{ij} = \begin{cases} 0 & \text{if } i \geq j \text{ (/)} \\ -\infty & \text{if } i < j \text{ } \end{cases}$$

$A + M$ softmax → 0 .


In [ ]:
#  
n = 6
mask = np.triu(np.ones((n, n)) * -1e9, k=1)  #   -inf
print("Causal Mask (  -inf):")
print(mask[:4, :4])

#  
np.random.seed(0)
scores = np.random.randn(n, n)  # Attention 
print(f"\nOriginal Scores ( ): {scores[0].round(2)}")
masked = scores + mask
print(f"Mask Application ( ): {masked[0].round(2)}")
weights = softmax(masked)
print(f"\n  ( ): {weights[0].round(4)}")
print(f"  →       (1.0)")

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(scores, cmap='viridis'); plt.colorbar(im0, ax=axes[0])
axes[0].set_title('Original Attention Scores')
im1 = axes[1].imshow(mask != -1e9, cmap='binary'); plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Causal Mask (=)')
im2 = axes[2].imshow(weights, cmap='Blues'); plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Mask Application  Weight')
plt.tight_layout()
plt.savefig('../figures/ch14_causal_mask.png', dpi=100, bbox_inches='tight')
plt.show()


## 14.6 [CPU/GPU ベンチマーク ⑤] シーケンス長 Attention 時間

シーケンス長 $n$ 2 Attention 時間 4 ( 4).
CPU vs GPU .


In [ ]:
# Attention ベンチマーク
import time
from llm_math.bench import time_fn, format_results_table

def attention_torch(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# シーケンス長
print(f"{'n':>6} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
print("-" * 50)
for n in [128, 256, 512, 1024, 2048]:
    d = 64
    Q = torch.randn(n, d)
    K = torch.randn(n, d)
    V = torch.randn(n, d)
    res_cpu = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=3)
    if torch.cuda.is_available():
        Q_g, K_g, V_g = Q.cuda(), K.cuda(), V.cuda()
        res_gpu = time_fn(attention_torch, Q_g, K_g, V_g, device='cuda', warmup=3, repeat=5)
        speedup = res_cpu['mean_ms'] / res_gpu['mean_ms']
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f} | {speedup:>9.2f}x")
    else:
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {'N/A':>12} | {'N/A':>10}")

if not torch.cuda.is_available():
    print("\n⚠ GPU . n=2048 複雑度Plane CPU   .")


In [ ]:
# PyTorch SDPA (scaled_dot_product_attention) 
print("PyTorch SDPA Function :")
print("( Flash Attention  Optimization   Line)\n")

n, d = 512, 64
Q = torch.randn(1, n, d)
K = torch.randn(1, n, d)
V = torch.randn(1, n, d)

#  
out_manual = attention_torch(Q, K, V)

# PyTorch SDPA (F.scaled_dot_product_attention)
out_sdpa = F.scaled_dot_product_attention(Q, K, V)

print(f" Implementation vs SDPA Maximum Error: {(out_manual - out_sdpa).abs().max():.2e}")
print("\n=>  Result. SDPA  Optimization .")

# 速度 比較
print("\nSpeed Comparison (n=512):")
t_manual = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=5)
def sdpa_call(Q, K, V):
    return F.scaled_dot_product_attention(Q, K, V)
t_sdpa = time_fn(sdpa_call, Q, K, V, device='cpu', warmup=2, repeat=5)
print(f"   Implementation: {t_manual['mean_ms']:.3f} ms")
print(f"  PyTorch SDPA: {t_sdpa['mean_ms']:.3f} ms")
print(f"  Speed Improvement: {t_manual['mean_ms'] / t_sdpa['mean_ms']:.2f}x")


## 14.7 Attention バックプロパゲーション 

 . :

$$\frac{\partial \mathcal{L}}{\partial Q} = \frac{1}{\sqrt{d_k}} \left(\frac{\partial \mathcal{L}}{\partial A} V K^\top + \ldots\right)$$

PyTorch `autograd` . Flash Attention バックプロパゲーション (Ch 27).

## 14.8 要点

| | | |
|---|---|---|
| Attention | $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$ | - 複雑度 値 |
| | $\frac{1}{\sqrt{d_k}}$ | 1 |
| | $M_{ij} = -\infty$ if $i < j$ | |
| 複雑度 | $O(n^2 d)$ 時間, $O(n^2)$ 空間 | シーケンス長 2 |

## 演習問題

1. $\mathrm{Var}(QK^\top_{ij}) = d_k$ Q, K iid $\mathcal{N}(0,1)$ 複雑度.
2. Attention Attention 出力 比較.
3. Attention 行列 可視化, .
4. シーケンス長 256, 512, 1024, 2048 CPU 時間 $O(n^2)$ 検証.
5. PyTorch SDPA `is_causal=True` Attention .

> 解答: `solutions/ch14_solutions.ipynb`
